In [1]:
# SETUP - Self-contained, works anywhere

import pandas as pd
import numpy as np
import sqlite3
import os
import json
import requests
from datetime import datetime, timedelta
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Auto-detect environment
if os.path.exists('/workspaces/quantum-ai-trader_v1.1'):
    WORKSPACE = '/workspaces/quantum-ai-trader_v1.1'
    ENV = 'Codespace'
elif os.path.exists(r'C:\Users\Shadow\quantum-ai-trader_v1.1'):
    WORKSPACE = r'C:\Users\Shadow\quantum-ai-trader_v1.1'
    ENV = 'Shadow PC'
else:
    WORKSPACE = str(Path.cwd().parent)
    ENV = 'JupyterLab GPU'

DATA_DIR = os.path.join(WORKSPACE, 'data')
DB_PATH = os.path.join(DATA_DIR, 'trading_system.db')

# Load API keys
from dotenv import load_dotenv
load_dotenv(os.path.join(WORKSPACE, '.env'))
PERPLEXITY_API_KEY = os.getenv('PERPLEXITY_API_KEY')

print(f"✅ Environment: {ENV}")
print(f"📁 Database: {DB_PATH}")
print(f"🔑 Perplexity API: {'✅ Loaded' if PERPLEXITY_API_KEY else '❌ Missing'}")
print(f"🕐 Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Environment: Shadow PC
📁 Database: C:\Users\Shadow\quantum-ai-trader_v1.1\data\trading_system.db
🔑 Perplexity API: ❌ Missing
🕐 Started: 2025-12-15 00:12:56


In [2]:
# SETUP - Self-contained, works anywhere

import pandas as pd
import numpy as np
import sqlite3
import os
import json
import requests
from datetime import datetime, timedelta
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Auto-detect environment
if os.path.exists('/workspaces/quantum-ai-trader_v1.1'):
    WORKSPACE = '/workspaces/quantum-ai-trader_v1.1'
    ENV = 'Codespace'
elif os.path.exists(r'C:\Users\Shadow\quantum-ai-trader_v1.1'):
    WORKSPACE = r'C:\Users\Shadow\quantum-ai-trader_v1.1'
    ENV = 'Shadow PC'
else:
    WORKSPACE = str(Path.cwd().parent)
    ENV = 'JupyterLab GPU'

DATA_DIR = os.path.join(WORKSPACE, 'data')
DB_PATH = os.path.join(DATA_DIR, 'trading_system.db')

# Load API keys
from dotenv import load_dotenv
load_dotenv(os.path.join(WORKSPACE, '.env'))
PERPLEXITY_API_KEY = os.getenv('PERPLEXITY_API_KEY')

print(f"✅ Environment: {ENV}")
print(f"📁 Database: {DB_PATH}")
print(f"🔑 Perplexity API: {'✅ Loaded' if PERPLEXITY_API_KEY else '❌ Missing'}")
print(f"🕐 Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Environment: Shadow PC
📁 Database: C:\Users\Shadow\quantum-ai-trader_v1.1\data\trading_system.db
🔑 Perplexity API: ❌ Missing
🕐 Started: 2025-12-15 00:14:12


In [3]:
# DATABASE CONNECTION

conn = sqlite3.connect(DB_PATH)

# Quick sanity check
check_query = """
    SELECT COUNT(DISTINCT ticker) as tickers,
           MIN(date) as earliest,
           MAX(date) as latest,
           COUNT(*) as total_bars
    FROM ohlcv_daily
"""
stats = pd.read_sql_query(check_query, conn)
print(f"📊 Database: {stats['tickers'].iloc[0]} tickers, {stats['total_bars'].iloc[0]:,} bars")
print(f"📅 Range: {stats['earliest'].iloc[0]} to {stats['latest'].iloc[0]}")

if stats['tickers'].iloc[0] < 300:
    print("⚠️  WARNING: Less than 300 tickers - data collection incomplete?")

📊 Database: 0 tickers, 0 bars
📅 Range: None to None
⚠️  WARNING: Less than 300 tickers - data collection incomplete?


In [4]:
# SETUP - Self-contained, works anywhere

import pandas as pd
import numpy as np
import sqlite3
import os
import json
import requests
from datetime import datetime, timedelta
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Auto-detect environment
if os.path.exists('/workspaces/quantum-ai-trader_v1.1'):
    WORKSPACE = '/workspaces/quantum-ai-trader_v1.1'
    ENV = 'Codespace'
elif os.path.exists(r'C:\Users\Shadow\quantum-ai-trader_v1.1'):
    WORKSPACE = r'C:\Users\Shadow\quantum-ai-trader_v1.1'
    ENV = 'Shadow PC'
else:
    WORKSPACE = str(Path.cwd().parent)
    ENV = 'JupyterLab GPU'

DATA_DIR = os.path.join(WORKSPACE, 'data')
DB_PATH = os.path.join(DATA_DIR, 'trading_system.db')

# Load API keys
from dotenv import load_dotenv
load_dotenv(os.path.join(WORKSPACE, '.env'))
PERPLEXITY_API_KEY = os.getenv('PERPLEXITY_API_KEY')

print(f"✅ Environment: {ENV}")
print(f"📁 Database: {DB_PATH}")
print(f"🔑 Perplexity API: {'✅ Loaded' if PERPLEXITY_API_KEY else '❌ Missing'}")
print(f"🕐 Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Environment: Shadow PC
📁 Database: C:\Users\Shadow\quantum-ai-trader_v1.1\data\trading_system.db
🔑 Perplexity API: ❌ Missing
🕐 Started: 2025-12-15 00:16:57


In [5]:
# DATABASE CONNECTION

conn = sqlite3.connect(DB_PATH)

# Quick sanity check
check_query = """
    SELECT COUNT(DISTINCT ticker) as tickers,
           MIN(date) as earliest,
           MAX(date) as latest,
           COUNT(*) as total_bars
    FROM ohlcv_daily
"""
stats = pd.read_sql_query(check_query, conn)
print(f"📊 Database: {stats['tickers'].iloc[0]} tickers, {stats['total_bars'].iloc[0]:,} bars")
print(f"📅 Range: {stats['earliest'].iloc[0]} to {stats['latest'].iloc[0]}")

# KILL SWITCH: Database must have data
if stats['tickers'].iloc[0] == 0:
    print("\n" + "="*80)
    print("❌ CRITICAL: Database is empty!")
    print("="*80)
    print("\nYou have two options:\n")
    print("Option 1 (RECOMMENDED - Run on Shadow PC with GPU):")
    print("  1. Open DAY1_DATA_COLLECTION.ipynb")
    print("  2. Run all cells (self-contained, no installs)")
    print("  3. Takes 3-4 hours, collects 353 tickers × 2 years")
    print("  4. Then come back and run this notebook\n")
    print("Option 2 (Quick - Copy from Codespace):")
    print("  1. On Codespace terminal:")
    print("     tar -czf trading_db.tar.gz data/trading_system.db")
    print("  2. Download trading_db.tar.gz to Shadow PC")
    print(f"  3. Extract to: {DATA_DIR}")
    print("  4. Re-run this cell")
    print("\n" + "="*80)
    raise SystemExit("Database empty - follow instructions above")

if stats['tickers'].iloc[0] < 300:
    print("⚠️  WARNING: Less than 300 tickers - data collection incomplete?")

📊 Database: 0 tickers, 0 bars
📅 Range: None to None

❌ CRITICAL: Database is empty!

You have two options:

Option 1 (RECOMMENDED - Run on Shadow PC with GPU):
  1. Open DAY1_DATA_COLLECTION.ipynb
  2. Run all cells (self-contained, no installs)
  3. Takes 3-4 hours, collects 353 tickers × 2 years
  4. Then come back and run this notebook

Option 2 (Quick - Copy from Codespace):
  1. On Codespace terminal:
     tar -czf trading_db.tar.gz data/trading_system.db
  2. Download trading_db.tar.gz to Shadow PC
  3. Extract to: C:\Users\Shadow\quantum-ai-trader_v1.1\data
  4. Re-run this cell



SystemExit: Database empty - follow instructions above

In [6]:
# SELF-CONTAINED - Works on Codespace AND Shadow PC
# No external imports - everything embedded

import yfinance as yf
import pandas as pd
import numpy as np
import time
import json
import sqlite3
import os
from datetime import datetime, timedelta
from pathlib import Path
from IPython.display import display, HTML, clear_output

# Auto-detect environment (Codespace vs Shadow PC)
if os.path.exists('/workspaces/quantum-ai-trader_v1.1'):
    WORKSPACE = '/workspaces/quantum-ai-trader_v1.1'
    ENV = 'Codespace'
elif os.path.exists(r'C:\Users\Shadow\quantum-ai-trader_v1.1'):
    WORKSPACE = r'C:\Users\Shadow\quantum-ai-trader_v1.1'
    ENV = 'Shadow PC'
else:
    # Fallback to notebook's parent directory
    WORKSPACE = str(Path.cwd().parent)
    ENV = 'Unknown'

DATA_DIR = os.path.join(WORKSPACE, 'data')
DB_PATH = os.path.join(DATA_DIR, 'trading_system.db')
CHECKPOINT_FILE = os.path.join(DATA_DIR, 'ohlcv_checkpoint.json')

# Ensure directories exist
os.makedirs(DATA_DIR, exist_ok=True)

print(f"✅ Environment: {ENV}")
print(f"📁 Workspace: {WORKSPACE}")
print(f"💾 Database: {DB_PATH}")
print(f"🕐 Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Environment: Shadow PC
📁 Workspace: C:\Users\Shadow\quantum-ai-trader_v1.1
💾 Database: C:\Users\Shadow\quantum-ai-trader_v1.1\data\trading_system.db
🕐 Started: 2025-12-15 00:17:41


In [7]:
# DATABASE CLASS - EMBEDDED (no external imports needed)

class TradingDatabase:
    """Production database - embedded in notebook for portability."""
    
    def __init__(self, db_path=DB_PATH):
        self.db_path = db_path
        self.conn = None
        
    def connect(self):
        """Connect to database with optimizations."""
        self.conn = sqlite3.connect(self.db_path, timeout=30.0)
        self.conn.execute("PRAGMA journal_mode=WAL")
        self.conn.execute("PRAGMA synchronous=NORMAL")
        self.conn.execute("PRAGMA cache_size=-64000")
        return self.conn
    
    def close(self):
        if self.conn:
            self.conn.close()
            self.conn = None
    
    def create_schema(self):
        """Create database tables."""
        cursor = self.conn.cursor()
        
        # Tickers table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS tickers (
                ticker TEXT PRIMARY KEY,
                sector TEXT,
                industry TEXT,
                market_cap_category TEXT,
                notes TEXT,
                active INTEGER DEFAULT 1,
                added_date TEXT DEFAULT CURRENT_TIMESTAMP
            )
        """)
        
        # OHLCV table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS ohlcv_daily (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                ticker TEXT NOT NULL,
                date TEXT NOT NULL,
                open REAL, high REAL, low REAL, close REAL,
                volume INTEGER, adj_close REAL,
                created_at TEXT DEFAULT CURRENT_TIMESTAMP,
                UNIQUE(ticker, date)
            )
        """)
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_ohlcv_ticker_date ON ohlcv_daily(ticker, date)")
        
        self.conn.commit()
    
    def load_ticker_universe(self, csv_path):
        """Load tickers from CSV - handles any CSV format."""
        if not os.path.exists(csv_path):
            print(f"⚠️ CSV not found at {csv_path}")
            return 0
        
        df = pd.read_csv(csv_path)
        
        # Ensure 'ticker' column exists
        if 'ticker' not in df.columns:
            # Try common alternatives
            if 'symbol' in df.columns:
                df = df.rename(columns={'symbol': 'ticker'})
            elif 'Ticker' in df.columns:
                df = df.rename(columns={'Ticker': 'ticker'})
            else:
                print(f"❌ No ticker column found in CSV")
                return 0
        
        # Add missing columns with defaults
        if 'sector' not in df.columns:
            df['sector'] = 'Unknown'
        if 'industry' not in df.columns:
            df['industry'] = 'Unknown'
        if 'market_cap_category' not in df.columns:
            df['market_cap_category'] = 'Unknown'
        if 'notes' not in df.columns:
            df['notes'] = ''
        if 'active' not in df.columns:
            df['active'] = 1
        if 'added_date' not in df.columns:
            df['added_date'] = datetime.now().isoformat()
        
        # Keep only needed columns
        cols = ['ticker', 'sector', 'industry', 'market_cap_category', 'notes', 'active', 'added_date']
        df = df[cols]
        
        # Insert into database
        df.to_sql('tickers', self.conn, if_exists='replace', index=False)
        self.conn.commit()
        
        return len(df)
    
    def get_active_tickers(self):
        """Get all active tickers."""
        query = "SELECT ticker FROM tickers WHERE active = 1 ORDER BY ticker"
        return pd.read_sql_query(query, self.conn)['ticker'].tolist()
    
    def insert_ohlcv_batch(self, ticker, ohlcv_df):
        """Insert OHLCV data."""
        if ohlcv_df.empty:
            return 0
        
        ohlcv_df = ohlcv_df.reset_index()
        ohlcv_df['ticker'] = ticker
        ohlcv_df['date'] = pd.to_datetime(ohlcv_df['Date']).dt.strftime('%Y-%m-%d')
        ohlcv_df['created_at'] = datetime.now().isoformat()
        
        col_mapping = {
            'Open': 'open', 'High': 'high', 'Low': 'low',
            'Close': 'close', 'Volume': 'volume', 'Adj Close': 'adj_close'
        }
        ohlcv_df = ohlcv_df.rename(columns=col_mapping)
        
        cols = ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume', 'adj_close', 'created_at']
        insert_df = ohlcv_df[cols]
        
        insert_df.to_sql('ohlcv_daily', self.conn, if_exists='append', index=False)
        self.conn.commit()
        return len(insert_df)
    
    def get_latest_ohlcv_date(self, ticker):
        """Get most recent date for ticker."""
        query = "SELECT MAX(date) as max_date FROM ohlcv_daily WHERE ticker = ?"
        result = pd.read_sql_query(query, self.conn, params=[ticker])
        return result['max_date'].iloc[0] if not result.empty else None
    
    def get_database_stats(self):
        """Get database statistics."""
        cursor = self.conn.cursor()
        stats = {}
        
        cursor.execute("SELECT COUNT(*) FROM tickers WHERE active = 1")
        stats['active_tickers'] = cursor.fetchone()[0]
        
        cursor.execute("""
            SELECT COUNT(DISTINCT ticker) as tickers_with_data,
                   MIN(date) as earliest_date,
                   MAX(date) as latest_date,
                   COUNT(*) as total_bars
            FROM ohlcv_daily
        """)
        row = cursor.fetchone()
        stats['ohlcv_tickers'] = row[0]
        stats['ohlcv_earliest'] = row[1]
        stats['ohlcv_latest'] = row[2]
        stats['ohlcv_total_bars'] = row[3]
        
        return stats

# Initialize database
db = TradingDatabase()
db.connect()
db.create_schema()

# Load ticker universe
UNIVERSE_FILE = os.path.join(DATA_DIR, 'ticker_universe_300.csv')
ticker_count = db.load_ticker_universe(UNIVERSE_FILE)

# Get all tickers
all_tickers = db.get_active_tickers()
print(f"\n📊 Total universe: {len(all_tickers)} tickers")
print(f"Sample: {all_tickers[:20]}")

# Check current database state
stats = db.get_database_stats()
print(f"\n📁 Current database state:")
print(f"   Tickers with data: {stats['ohlcv_tickers']}")
print(f"   Total bars: {stats['ohlcv_total_bars']:,}")

if stats['ohlcv_tickers'] > 0:
    print(f"   Date range: {stats['ohlcv_earliest']} to {stats['ohlcv_latest']}")


📊 Total universe: 353 tickers
Sample: ['AAPL', 'ABCL', 'ABNB', 'ACHR', 'ADBE', 'ADPT', 'AEVA', 'AFRM', 'AI', 'AIXI', 'AKRO', 'AKYA', 'ALB', 'ALKT', 'ALLO', 'ALNY', 'AMBA', 'AMD', 'AMPL', 'AMSC']

📁 Current database state:
   Tickers with data: 0
   Total bars: 0


In [8]:
# Checkpoint management
CHECKPOINT_FILE = '/workspaces/quantum-ai-trader_v1.1/data/ohlcv_checkpoint.json'

def load_checkpoint():
    """Load previous progress."""
    try:
        with open(CHECKPOINT_FILE, 'r') as f:
            checkpoint = json.load(f)
            return set(checkpoint['processed']), checkpoint.get('failed', [])
    except FileNotFoundError:
        return set(), []
    except Exception as e:
        print(f"⚠️ Checkpoint load error: {e}")
        return set(), []

def save_checkpoint(processed_set, failed_list, success_count, total_count):
    """Save current progress."""
    checkpoint = {
        'timestamp': datetime.now().isoformat(),
        'processed': list(processed_set),
        'failed': failed_list,
        'success_count': success_count,
        'total_processed': total_count
    }
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f, indent=2)

# Load existing checkpoint
processed_set, failed_list = load_checkpoint()
print(f"📁 Checkpoint loaded: {len(processed_set)} already processed")

remaining = [t for t in all_tickers if t not in processed_set]
print(f"🔄 Remaining to process: {len(remaining)} tickers")

📁 Checkpoint loaded: 0 already processed
🔄 Remaining to process: 353 tickers


In [ ]:
# MAIN DATA COLLECTION LOOP
# This runs for 3-4 hours - safe to walk away

print("="*60)
print("🚀 STARTING DATA COLLECTION")
print("="*60)
print(f"Total tickers: {len(all_tickers)}")
print(f"Remaining: {len(remaining)}")
print(f"Already done: {len(processed_set)}")
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

start_time = time.time()
success_count = len(processed_set)
total_processed = len(processed_set)

for i, ticker in enumerate(remaining, 1):
    total_processed += 1
    
    try:
        # Check if we already have recent data
        latest_date = db.get_latest_ohlcv_date(ticker)
        
        if latest_date:
            latest_dt = datetime.strptime(latest_date, '%Y-%m-%d')
            days_ago = (datetime.now() - latest_dt).days
            
            if days_ago < 7:
                print(f"[{i}/{len(remaining)}] {ticker} - up-to-date (last: {latest_date})")
                processed_set.add(ticker)
                success_count += 1
                time.sleep(0.2)
                continue
        
        # Download from yfinance
        data = yf.download(ticker, period='2y', progress=False, auto_adjust=False)
        
        if data.empty or len(data) < 20:
            print(f"[{i}/{len(remaining)}] {ticker} - ❌ No data ({len(data)} bars)")
            failed_list.append(ticker)
            continue
        
        # Insert into database
        rows = db.insert_ohlcv_batch(ticker, data)
        print(f"[{i}/{len(remaining)}] {ticker} - ✅ {rows} bars")
        
        processed_set.add(ticker)
        success_count += 1
        
    except Exception as e:
        print(f"[{i}/{len(remaining)}] {ticker} - ❌ Error: {str(e)[:50]}")
        failed_list.append(ticker)
    
    # Progress update every 10 tickers
    if i % 10 == 0:
        elapsed = time.time() - start_time
        rate = i / elapsed if elapsed > 0 else 0
        remaining_count = len(remaining) - i
        eta = remaining_count / rate if rate > 0 else 0
        
        print("\n" + "="*60)
        print(f"📊 PROGRESS: {len(processed_set)}/{len(all_tickers)} ({len(processed_set)/len(all_tickers)*100:.1f}%)")
        print(f"✅ Success: {success_count} | ❌ Failed: {len(failed_list)}")
        print(f"⏱️  Rate: {rate*60:.1f} tickers/min")
        print(f"🕐 ETA: {eta/60:.1f} min ({eta/3600:.2f} hours)")
        print(f"🕐 Current time: {datetime.now().strftime('%H:%M:%S')}")
        print("="*60 + "\n")
    
    # Save checkpoint every 50 tickers
    if i % 50 == 0:
        save_checkpoint(processed_set, failed_list, success_count, total_processed)
        print(f"\n📁 CHECKPOINT SAVED at {len(processed_set)} tickers\n")
    
    # Polite delay
    time.sleep(0.5)

# Final checkpoint
save_checkpoint(processed_set, failed_list, success_count, total_processed)

# Summary
elapsed_total = time.time() - start_time
print("\n" + "="*60)
print("✅ DATA COLLECTION COMPLETE")
print("="*60)
print(f"Total processed: {total_processed}")
print(f"Successful: {success_count}")
print(f"Failed: {len(failed_list)}")
print(f"Success rate: {success_count/total_processed*100:.1f}%")
print(f"Total runtime: {elapsed_total/60:.1f} min ({elapsed_total/3600:.2f} hours)")
print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

🚀 STARTING DATA COLLECTION
Total tickers: 353
Remaining: 353
Already done: 0

Started: 2025-12-15 00:18:31
[1/353] AAPL - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[2/353] ABCL - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[3/353] ABNB - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[4/353] ACHR - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[5/353] ADBE - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[6/353] ADPT - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[7/353] AEVA - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[8/353] AFRM - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[9/353] AI - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[10/353] AIXI - ❌ Error: table ohlcv_daily has no column named ('ticker', '

📊 PROGRESS: 0/353 (0.0%)
✅ Success: 0 | ❌ Failed: 10
⏱️  Rate: 101.3 tickers/min
🕐 ETA: 3.4 min (0.06 hours)
🕐 Current time: 00:18:37

[11/353]

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AKYA"}}}

1 Failed download:
['AKYA']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')


[12/353] AKYA - ❌ No data (0 bars)
[13/353] ALB - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[14/353] ALKT - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[15/353] ALLO - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[16/353] ALNY - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[17/353] AMBA - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[18/353] AMD - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[19/353] AMPL - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[20/353] AMSC - ❌ Error: table ohlcv_daily has no column named ('ticker', '

📊 PROGRESS: 0/353 (0.0%)
✅ Success: 0 | ❌ Failed: 20
⏱️  Rate: 99.8 tickers/min
🕐 ETA: 3.3 min (0.06 hours)
🕐 Current time: 00:18:43

[21/353] AMWL - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[22/353] AMZN - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[23/353] ANF - ❌ Error: table ohlcv_daily has no column named ('ticker',


1 Failed download:
['ARVL']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')


[29/353] ARVL - ❌ No data (0 bars)
[30/353] ARVN - ❌ Error: table ohlcv_daily has no column named ('ticker', '

📊 PROGRESS: 0/353 (0.0%)
✅ Success: 0 | ❌ Failed: 30
⏱️  Rate: 99.6 tickers/min
🕐 ETA: 3.2 min (0.05 hours)
🕐 Current time: 00:18:49

[31/353] ARWR - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[32/353] ASTS - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[33/353] AURA - ❌ Error: table ohlcv_daily has no column named ('ticker', '
[34/353] AUTL - ❌ Error: table ohlcv_daily has no column named ('ticker', '
